# Cluster map investigation

## Rotation check

Visual sanity check that the catalogue's **rotated** angular columns (`theta_rot_rad`, `phi_rot_rad`) correctly place clusters on the L2p8 y-map, which lives in the yang26-rotated frame.

The rotation is applied **per lightcone shell**. Low-redshift shells (`z < 0.35`, shell index $\le 6$) are effectively unrotated, so the native and rotated positions coincide. From `z ~ 0.35` (shell 7) onward the per-shell rotation differs from the identity by tens of degrees. We gnomview-zoom the most massive clusters in each regime at their **rotated** coordinates; a bright tSZ blob dead-centre in every panel confirms the rotation was applied correctly.

In [1]:
import numpy as np
import pandas as pd
import healpy as hp
import matplotlib.pyplot as plt

# Publication-quality plot defaults
plt.rcParams.update({
    "text.usetex": False,
    "font.family": "serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.dpi": 100,
    "savefig.dpi": 300,
    "axes.linewidth": 0.8,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
})


from flamingo import paths
from flamingo.maps import read_map
from flamingo.catalogue import load_catalogue

ymap = read_map(paths.HYDRO_MAP)
df = load_catalogue(paths.HYDRO_CATALOGUE)
print('NSIDE', hp.npix2nside(ymap.size), '| clusters', len(df))


NSIDE 4096 | clusters 1555542


In [2]:
# Most massive clusters in the local (unrotated) and distant (rotated) regimes.
local = df[df['z'] < 0.35].nlargest(3, 'M_500c_Msun')
distant = df[df['z'] > 1.10].nlargest(3, 'M_500c_Msun')
cols = ['z', 'shell_idx', 'M_500c_Msun', 'theta500_arcmin',
        'lon_rot_deg', 'lat_rot_deg', 'lon_nat_deg', 'lat_nat_deg']
print('LOCAL (z < 0.35)'); print(local[cols].to_string())
print('\nDISTANT (z > 1.10)'); print(distant[cols].to_string())

LOCAL (z < 0.35)
                z  shell_idx   M_500c_Msun  theta500_arcmin  lon_rot_deg  lat_rot_deg  lon_nat_deg  lat_nat_deg
1482157  0.264037          5  2.065920e+15         7.138118     63.85552    17.716590     63.85552    17.716590
1464932  0.313081          6  1.963520e+15         6.118454    153.84720     2.390363    153.84720     2.390363
1552300  0.093700          1  1.768960e+15        16.991050    208.20270   -60.661190    208.20270   -60.661190

DISTANT (z > 1.10)
               z  shell_idx   M_500c_Msun  theta500_arcmin  lon_rot_deg  lat_rot_deg  lon_nat_deg  lat_nat_deg
416258  1.119055         22  7.718400e+14         1.835343   -151.31950     26.41751     67.66037     23.33691
451455  1.101433         22  7.718400e+14         1.856091    162.30530     13.68949     67.65743    -21.78284
408101  1.170115         23  7.628800e+14         1.780567     76.35959    -32.08838    229.10960    -63.09092


In [3]:
def gnom(row, *, frame='rot', sub=None, title=None, fov_in_theta500=5.0):
    """gnomview a cluster from its catalogue row.

    frame='rot' uses the rotated columns (correct for this map);
    frame='nat' uses the native columns (should miss at high z).
    """
    lon = float(row[f'lon_{frame}_deg'])
    lat = float(row[f'lat_{frame}_deg'])
    xsize = 240
    reso = max(0.1, fov_in_theta500 * float(row['theta500_arcmin']) / xsize)
    hp.gnomview(ymap, rot=(lon, lat), reso=reso, xsize=xsize,
                min=0.0, cmap='viridis', notext=True, sub=sub,
                title=title)
    hp.graticule(dpar=reso * xsize / 6 / 60, dmer=reso * xsize / 6 / 60,
                 local=True, verbose=False)

## Local massive clusters (z < 0.35)

Native and rotated coordinates are identical here. Each panel should show a bright, centred tSZ source. The graticule crossing marks the catalogue position.

In [4]:
fig = plt.figure(figsize=(13, 4.2))
for i, (_, row) in enumerate(local.iterrows(), start=1):
    gnom(row, frame='rot', sub=(1, 3, i),
         title=f"z={row['z']:.2f}  M500={row['M_500c_Msun']:.2e}")
plt.show()

/home/lxu/miniconda3/envs/hmfast_py311/lib/python3.11/site-packages/healpy/visufunc.py:1630: HealpyDeprecationWarning: "verbose" was deprecated in version 1.15.0 and will be removed in a future version. 
  ax.graticule(dpar=dpar, dmer=dmer, coord=coord, local=local, **kwds)


## Distant massive clusters (z > 0.50)

Using the **rotated** columns at high redshift, where the per-shell yang26 rotation is non-trivial. Each cluster sits centred on a bright tSZ source, confirming the rotation was applied correctly.

In [5]:
fig = plt.figure(figsize=(13, 4.2))
for i, (_, row) in enumerate(distant.iterrows(), start=1):
    gnom(row, frame='rot', sub=(1, 3, i),
         title=f"z={row['z']:.2f}  M500={row['M_500c_Msun']:.2e}")
plt.show()

## Quantitative check

Map value at the rotated position for the distant clusters; each should sit on a strong y-peak well above the sky background.

In [6]:
nside = hp.npix2nside(ymap.size)
sky = float(np.median(ymap))
for _, row in distant.iterrows():
    p_rot = hp.ang2pix(nside, row['theta_rot_rad'], row['phi_rot_rad'])
    print(f"z={row['z']:.2f}  y(rot)={ymap[p_rot]:.3e}  "
          f"y(rot)/sky={ymap[p_rot]/sky:.1f}")
print('\nsky median y =', sky)

z=1.12  y(rot)=1.172e-04  y(rot)/sky=109.1
z=1.10  y(rot)=1.227e-04  y(rot)/sky=114.3
z=1.17  y(rot)=1.322e-04  y(rot)/sky=123.1

sky median y = 1.0742162710465954e-06


# Individual empirical y-profiles

We measure the tSZ (Compton-$y$) profile of individual clusters directly from the pixelized map using a **fully empirical** estimator, with **no** pressure-profile model and **no** central value $y_0$.

For each cluster we bin pixels in scaled radius $x=\theta/\theta_{500}$ (with $\theta_{500}$ from `theta500_arcmin`), take the annular mean $\bar y_i = \langle y_p\rangle_{i}$ (equal weights $w_p=1$), and normalise by the aperture-mean inside $R_{500}$,
$$y_{\rm norm} = \frac{Y_{500}}{\pi\theta_{500}^2},\qquad Y_{500}=\sum_{\theta_p<\theta_{500}} y_p\,\Omega_{\rm pix}.$$

We plot $g(x)=\bar y(x)/y_{\rm norm}$ versus $x$. The dashed curve is the Arnaud A10 projected GNFW shape normalised the **same** way (divided by its aperture mean inside $x<1$), so it is directly comparable. The estimator is defined in this notebook; we do not use `flamingo.profiles.stacking`.

In [7]:
from flamingo.geometry import query_disc_separation, ARCMIN_PER_RAD
from flamingo.profiles import projected_shape

OMEGA_PIX = hp.nside2pixarea(nside)   # solid angle per pixel [sr]

# Scaled-radius bins x = theta / theta_500, with area-weighted bin centres.
x_edges = np.logspace(np.log10(0.1), np.log10(4.0), 21)
x_lo, x_hi = x_edges[:-1], x_edges[1:]
x_mid = (2.0 / 3.0) * (x_hi**3 - x_lo**3) / (x_hi**2 - x_lo**2)


def empirical_profile(theta_c, phi_c, theta500_arcmin, x_edges):
    """Empirical per-cluster profile from the y-map: annular means (w_p=1) and
    the aperture-mean normalisation y_norm = Y500/(pi theta500^2). No y_0."""
    theta500_rad = theta500_arcmin / ARCMIN_PER_RAD
    pix, sep_rad = query_disc_separation(nside, theta_c, phi_c, x_edges[-1] * theta500_rad)
    y_p = ymap[pix]
    x = sep_rad / theta500_rad

    ybar = np.full(len(x_edges) - 1, np.nan)
    idx = np.digitize(x, x_edges) - 1
    for b in range(ybar.size):
        sel = idx == b
        if sel.any():
            ybar[b] = y_p[sel].mean()

    inside = x < 1.0
    y_norm = (y_p[inside].sum() * OMEGA_PIX) / (np.pi * theta500_rad**2)
    return ybar, y_norm


def aperture_normalised_model(x_grid, shape_func):
    """Model shape divided by its own aperture mean inside x<1 (same as data)."""
    f = np.asarray(shape_func(x_grid))
    xa = np.linspace(0.0, 1.0, 400)
    aperture_mean = 2.0 * np.trapezoid(np.asarray(shape_func(xa)) * xa, xa)
    return f / aperture_mean


sample = pd.concat([local, distant])
fig, ax = plt.subplots(figsize=(7.2, 5.6))
colors = plt.cm.viridis(np.linspace(0, 0.9, len(sample)))
for c, (_, row) in zip(colors, sample.iterrows()):
    ybar, y_norm = empirical_profile(
        float(row['theta_rot_rad']), float(row['phi_rot_rad']),
        float(row['theta500_arcmin']), x_edges)
    ax.plot(x_mid, ybar / y_norm, 'o-', ms=3, lw=1, color=c,
            label=f"z={row['z']:.2f}, M500={row['M_500c_Msun']:.1e}")

xx = np.logspace(np.log10(0.05), np.log10(4.0), 200)
ax.plot(xx, aperture_normalised_model(xx, projected_shape), 'k--', lw=1.8,
        label='Arnaud A10 (projected, aperture-norm)')

ax.axvline(1.0, color='grey', ls=':', lw=1)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel(r'$\theta / \theta_{500}$')
ax.set_ylabel(r'$g = \bar y / (Y_{500}/\pi\theta_{500}^2)$')
ax.set_ylim(1e-2, 3e1)
ax.set_title('FLAMINGO L2p8_m9: empirical aperture-normalised cluster profiles')
ax.legend(fontsize=8)
fig.tight_layout(); plt.show()

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
